3D setup to recreate the in-vivo EFOV study in-silico

Imports

In [ ]:
import json
import os
import shutil
import re
import sys
import time
from pathlib import Path
import numpy as np
from carputils import settings, tools
import igb_io

Mesh Resolution and Size

In [ ]:
# Slab mesh: 3.0 3.0 0.4
# Resolution 250 250 250 (um)
slabSize = (30.0, 30.0, 4.0)
resolution = 0.25

# Areas of the mesh ------ TAGS
wallTag = 1
bundleTag = 2
bundleExitTag = 3
sheathTag = 4
scarTag = 5

Mesh Building + Setup Functions

In [ ]:

def allMeshFilesExist(meshBaseName):
    for extension in [".pts", ".elem", ".lon"]:
        if not os.path.exists(meshBaseName + extension):
            return False
    return True
 
 
def readNodePositions(ptsFilePath):
    """Read the .pts file. Returns a list of [x, y, z] in micrometres."""
    nodePositions = []
    
    with open(ptsFilePath) as ptsFile:
        allLines = ptsFile.read().splitlines()

    for line in allLines[1:]:  # first line is just the node count
        if line.strip() == "":
            continue
        parts = line.split()
        nodePositions.append([float(parts[0]), float(parts[1]), float(parts[2])])
    return nodePositions
 
 
def readElements(elemFilePath):
    """Read the .elem file. Each element looks like ['Tt', n1, n2, n3, n4, tag]."""
    elementList = []
    with open(elemFilePath) as elemFile:
        allLines = elemFile.read().splitlines()
    for line in allLines[1:]:  # first line is just the element count
        if line.strip() == "":
            continue
        elementList.append(line.split())
    return elementList
 
 
def chooseTagForElement(centreX, centreY, centreZ):
    """Decide which tissue region an element belongs to, from its centre (mm)."""
    distanceFromMidlineY = abs(centreY)
    isWithinBundleLength = -7.5 < centreX < 7.5
 
    isInSheathSideWalls = 1.25 < distanceFromMidlineY < 1.5
    isInSheathFloor = centreZ < 0.5 and distanceFromMidlineY < 1.5
    isSheath = (
        isWithinBundleLength
        and centreZ > 0.25
        and (isInSheathSideWalls or isInSheathFloor)
    )
 
    isBundle = (
        isWithinBundleLength
        and distanceFromMidlineY < 1.25
        and centreZ > 0.5
    )
 
    if isBundle and centreX > 5.5:
        return bundleExitTag
    if isBundle:
        return bundleTag
    if isSheath:
        return sheathTag
    return wallTag
 
 
def buildMesh(job):
    meshFolder = os.path.join(job.ID, "mesh")
    preAblationMesh = os.path.join(meshFolder, "slab_bundle")
    postAblationMesh = os.path.join(meshFolder, "slab_bundle_scar")
 
    # Step 1: if both meshes were already built, reuse them
    if allMeshFilesExist(preAblationMesh) and allMeshFilesExist(postAblationMesh):
        return preAblationMesh, postAblationMesh
 
    # Step 2: run openCARP's mesher to make a plain rectangular block
    os.makedirs(meshFolder, exist_ok=True)
    rawMesh = os.path.join(meshFolder, "slab_raw")
 
    mesherCommand = [settings.execs.MESHER]
    for axisNumber in range(3):
        sizeInCm = slabSize[axisNumber] / 10.0
        resolutionInMicrometres = resolution * 1000.0
        mesherCommand.append("-size[" + str(axisNumber) + "]")
        mesherCommand.append(sizeInCm)
        mesherCommand.append("-resolution[" + str(axisNumber) + "]")
        mesherCommand.append(resolutionInMicrometres)
    mesherCommand.append("-mesh")
    mesherCommand.append(rawMesh)
    job.bash(mesherCommand)
 
    # Step 3: check the mesher actually wrote its files
    missingFiles = []
    for extension in [".pts", ".elem", ".lon"]:
        if not os.path.isfile(rawMesh + extension):
            missingFiles.append(rawMesh + extension)
    if len(missingFiles) > 0:
        raise RuntimeError(
            "mesher did not write " + str(missingFiles)
            + "; check settings.execs.MESHER in your carputils settings"
        )
 
    # Step 4: check the block is centred on (0, 0, 0)
    nodePositions = readNodePositions(rawMesh + ".pts")
    for axisNumber in range(3):
        allValuesOnAxis = [position[axisNumber] for position in nodePositions]
        smallestMm = min(allValuesOnAxis) / 1000.0
        largestMm = max(allValuesOnAxis) / 1000.0
        expectedHalfSize = slabSize[axisNumber] / 2
        if abs(smallestMm + expectedHalfSize) > 0.01 or abs(largestMm - expectedHalfSize) > 0.01:
            raise ValueError(
                "mesher block is not centred on the origin on axis "
                + str(axisNumber) + ": " + str(smallestMm) + " to " + str(largestMm) + " mm"
            )
 
    # Step 5: read the elements and check they are all tetrahedra
    elementList = readElements(rawMesh + ".elem")
    for element in elementList:
        if element[0] != "Tt" or len(element) != 6:
            raise ValueError("Expected tagged tetrahedra from mesher")
 
    # Step 6: give every element a tag based on where its centre is
    preAblationTags = []
    postAblationTags = []
    for element in elementList:
        cornerNodeNumbers = [int(element[1]), int(element[2]), int(element[3]), int(element[4])]
 
        sumX = 0.0
        sumY = 0.0
        sumZ = 0.0
        for nodeNumber in cornerNodeNumbers:
            sumX += nodePositions[nodeNumber][0]
            sumY += nodePositions[nodeNumber][1]
            sumZ += nodePositions[nodeNumber][2]
        centreX = sumX / 4 / 1000.0  # micrometres -> mm
        centreY = sumY / 4 / 1000.0
        centreZ = sumZ / 4 / 1000.0
 
        tagBeforeAblation = chooseTagForElement(centreX, centreY, centreZ)
        preAblationTags.append(tagBeforeAblation)
 
        # After ablation: anything within the scar radius of the centre becomes scar
        distanceFromSlabCentre = math.hypot(centreX, centreY)
        if distanceFromSlabCentre < SCAR_R_MM:
            postAblationTags.append(scarTag)
        else:
            postAblationTags.append(tagBeforeAblation)
 
    # Step 7: write the two meshes (same nodes and fibres, different tags)
    for meshBaseName, tagList in [(preAblationMesh, preAblationTags), (postAblationMesh, postAblationTags)]:
        with open(meshBaseName + ".elem", "w") as elemFile:
            elemFile.write(str(len(elementList)) + "\n")
            for elementIndex in range(len(elementList)):
                element = elementList[elementIndex]
                lineWithoutOldTag = " ".join(element[0:5])
                elemFile.write(lineWithoutOldTag + " " + str(tagList[elementIndex]) + "\n")
        shutil.copy(rawMesh + ".pts", meshBaseName + ".pts")
        shutil.copy(rawMesh + ".lon", meshBaseName + ".lon")
 
    # Step 8: print a summary
    tagCounts = {}
    for tagValue in preAblationTags:
        tagCounts[tagValue] = tagCounts.get(tagValue, 0) + 1
    print("Mesh tags (pre): " + str(dict(sorted(tagCounts.items()))))
    print("Scar elements: " + str(postAblationTags.count(scarTag)))
 
    return preAblationMesh, postAblationMesh

EGM Setup + Electrode Layout

In [ ]:
egmGaps = (0.25, 1.0)
egmLattice = np.arange(-11, 12)  # 23 x 23 positions, 1 mm pitch
egmSamples = 8  # surface points per cylindrical electrode, averaged
egmSurfaceMM = 2.0
egmRadiusMM = 0.5

gridOffsetsMM = (-6, -2, 2, 6)  # 4 mm pitch, electrode e = row*4 + col (col along x)
scarRMM = 2.5  # lesion radius; d_mm = scarRMM - r (> 0 inside the lesion)

Electrode Functions

In [ ]:
def writeElectrodes(path="electrodes/egm_points.pts"):
    """Return and write recovery sample coordinates in micrometres."""
    points = []
    for gap in egmGaps:
        centre_z = egmSurfaceMM + gap + egmRadiusMM
        for x in egmLattice:
            for y in egmLattice:
                for dx in (-egmRadiusMM / 2, egmRadiusMM / 2):
                    for angle in np.deg2rad([0, 90, 180, 270]):
                        points.append(
                            [
                                x + dx,
                                y + egmRadiusMM * np.cos(angle),
                                centre_z + egmRadiusMM * np.sin(angle),
                            ]
                        )
    points_um = np.asarray(points) * 1000
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(path, points_um, fmt="%.1f", header=str(len(points_um)), comments="")
    return points_um

def writeRecoveryPoints(job):
    base = os.path.abspath(os.path.join(job.ID, "electrodes", "egm_points"))

    if not os.path.exists(base + ".pts"):
        writeElectrodes()

    if not os.path.isfile(base + ".pts"):
        raise RuntimeError("Electrodes did not write " + base + ".pts")
        
    return base

Scenarios

In [ ]:
# Stages per scenario: (stage .par file, output folder, folder whose checkpoint it restarts from)
SCENARIOS = {
    "pre": (
        False,
        [
            ("s1_pace", "s1_pace", None)
        ],
    ),
    "ablate": (True, [("s4_ablate_terminate", "s4_ablate_terminate", "s1_pace")])
}

Error Checking + Handle

In [ ]:
if not np.isfinite(args.t_lift) or not 1802 < args.t_lift < 3500:
    raise ValueError(
        "--t-lift must be after the final stimulus (1802) and before the s3 checkpoint (3500 ms)"
    )
if args.tune_cv is not None:
    raise ValueError(
        "--tune-cv is disabled: the existing tuner omits cAF remodelling. Use validated --gi/--ge or common.par."
    )
if args.gi is not None and (not np.isfinite(args.gi) or args.gi <= 0):
    raise ValueError("--gi must be positive and finite")
if args.ge is not None and (not np.isfinite(args.ge) or args.ge <= 0):
    raise ValueError("--ge must be positive and finite")
if settings.platform.BATCH or getattr(args, "dry", False):
    raise ValueError(
        "This checkpoint chain needs immediate execution; use a local carputils platform (inside your compute allocation), without --dry"
    )

# Fail now, not after hours of simulation, if the simulator executables are misconfigured.
for name in ("MESHER", "CARP"):
    exe = getattr(settings.execs, name, None)
    if not exe or not (os.access(str(exe), os.X_OK) or shutil.which(str(exe))):
        raise RuntimeError(
            f"carputils settings.execs.{name} = {exe!r} is not an executable; "
            "set it in your carputils settings.yaml"
        )

# Conductivity overrides must be a complete intracellular/extracellular pair.
if (args.gi is None) != (args.ge is None):
    raise ValueError("Give both --gi and --ge, or neither for common.par")


Runner

In [ ]:

os.makedirs(job.ID, exist_ok=True)
mesh_pre, mesh_post = buildMesh(job)
egm_points = writeRecoveryPoints(job)

scenarios = list(SCENARIOS)

resume = getattr(args, "resume", False)
if not resume:
    for scenario in scenarios:
        for _, out, _ in SCENARIOS[scenario][1]:
            destination = Path(job.ID) / out
            if destination.exists() and any(destination.iterdir()):
                raise FileExistsError(
                    f"{destination} already contains output; archive it, use RESUME=1 "
                    "or SKIP_SIMS=1")

rerun = set()  # stages (re)simulated in this call: anything restarting from them is stale
for scenario in scenarios:
    scar, stages = SCENARIOS[scenario]
    for stage, out, restart in stages:
        simdir = os.path.join(job.ID, out)
        t0, tend = stageTimes(stage, args.t_lift)
        if resume and os.path.isdir(simdir) and os.listdir(simdir):
            if restart not in rerun and stageComplete(simdir, stage, args.t_lift):
                print("=== {}: {} complete, reusing {}".format(scenario, stage, simdir))
                continue
            aside = "{}.{}-{}".format(
                simdir,
                "stale" if restart in rerun else "incomplete",
                time.strftime("%Y%m%d-%H%M%S"),
            )
            os.rename(simdir, aside)
            print("=== {}: moved previous {} output to {}".format(scenario, stage, aside))
        rerun.add(out)

        # Layered settings: later files override earlier ones
        cmd = tools.carp_cmd(os.path.join(EXAMPLE_DIR, "common.par"))
        if scar:
            cmd += ["+F", os.path.join(EXAMPLE_DIR, "scar.par")]
        cmd += ["+F", os.path.join(EXAMPLE_DIR, stage + ".par")]

        # Per-run values (override the .par files)
        cmd += [
            "-simID",
            simdir,
            "-meshname",
            mesh_post if scar else mesh_pre,
            "-phie_rec_ptf",
            egm_points,
        ]
        if restart:
            state, _ = checkpoint(os.path.join(job.ID, restart), t0)
            cmd += ["-start_statef", state]
        if stage == "s2_block":
            cmd += ["-tend", args.t_lift, "-tsav[0]", args.t_lift]
        if args.gi is not None:
            cmd += conductivityOpts(args.gi, args.ge, stage)

        print("=== {}: {} -> {}  ({:g}-{:g} ms)".format(scenario, stage, simdir, t0, tend))
        started = time.time()
        job.carp(cmd)
        elapsed = time.time() - started

        required = ["vm.igb", "phie_recovery.igb", "parameters.par"]
        missing = [name for name in required if not (Path(simdir) / name).is_file()]
        if missing:
            raise RuntimeError(f"{out}: missing simulation outputs: {missing}")
        t_ms = stage_time_axis(Path(simdir) / "vm.igb", t0, tend)
        needed = CHECKPOINT_FOR.get(stage)
        if needed is not None:
            checkpoint(simdir, args.t_lift if needed == "t_lift" else needed)

        with open(os.path.join(simdir, "meshname.txt"), "w") as f:
            f.write(os.path.abspath(mesh_post if scar else mesh_pre) + "\n")

        # Unipolar electrograms from the recovered potentials
        buildEGMs(simdir, t0, egm_points, t_ms)

        # Metadata for the shared pipeline; written last, so a
        # stage with meta.json "complete" has every output
        with open(os.path.join(simdir, "meta.json"), "w") as f:
            json.dump(
                {
                    "model": "slab3d",
                    "run": os.path.basename(job.ID) + "/" + out,
                    "state": "post" if scar else "pre",
                    "scenario": scenario,
                    "stage": stage,
                    "t_start_ms": float(t_ms[0]),
                    "t_end_ms": float(t_ms[-1]),
                    "t_lift_ms": float(args.t_lift),
                    "restart_from": restart,
                    "np": getattr(args, "np", None),
                    "wall_time_s": round(elapsed, 1),
                    "complete": True,
                },
                f,
                indent=1,
            )
        print("=== {}: {} finished in {:.1f} min".format(scenario, stage, elapsed / 60))

# Do visualization: Vm of the last stage run
if args.visualize and not settings.platform.BATCH:
    job.meshalyzer(mesh_post if scar else mesh_pre, os.path.join(simdir, "vm.igb"))